# SVM и ядра: как меняется граница решения

Линейный SVM, `RBF`-ядро и влияние параметров `C` и `gamma`
рассматриваются на нескольких простых наборах данных.

## SVM

`Support Vector Machine` ищет гиперплоскость
$\mathbf{w}^T \mathbf{x} + b = 0$, которая держит максимальный
зазор между классами.

**Margin** равен $\dfrac{2}{\|\mathbf{w}\|}$, поэтому максимизация зазора
сводится к контролю нормы весов.

### Soft Margin

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \max(0, 1 - y_i(\mathbf{w}^T \mathbf{x}_i + b))$$

Слагаемое $\max(0, 1 - y_i f(\mathbf{x}_i))$ — это `Hinge Loss`.

### Что делает параметр `C`
- малый `C` разрешает больше ошибок, но обычно расширяет margin;
- большой `C` сильнее штрафует ошибки и может сделать границу жёстче.

### Kernel Trick

Вместо явного перехода в пространство высокой размерности используем kernel-функцию:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \langle \phi(\mathbf{x}_i), \phi(\mathbf{x}_j) \rangle$$

Для `RBF`:
$$K(\mathbf{x}, \mathbf{x}') = \exp\left(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2\right)$$

### Support Vectors
Это точки на границе margin или внутри неё. Именно они двигают решение;
далёкие от границы объекты на положение гиперплоскости уже почти не влияют.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "svm_kernels"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


 

np.random.seed(42)
print("Все библиотеки загружены успешно")

## Линейный SVM вручную через Hinge Loss

In [ ]:
class LinearSVM:
    """Линейный SVM через Hinge Loss и градиентный спуск."""
    
    def __init__(self, C=1.0, lr=0.001, n_iters=1000):
        self.C = C          # параметр регуляризации
        self.lr = lr        # learning rate
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.losses = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Метки должны быть +1 и -1
        y_ = np.where(y <= 0, -1, 1)
        
        # Инициализация весов
        self.w = np.zeros(n_features)
        self.b = 0
        
        lam = 1.0 / (self.C * n_samples)  # lambda = 1/(C*n)
        
        for epoch in range(self.n_iters):
            total_loss = 0
            
            # Перемешиваем данные (SGD)
            idx = np.random.permutation(n_samples)
            
            for i in idx:
                margin = y_[i] * (np.dot(X[i], self.w) + self.b)
                
                # Hinge loss для текущей точки
                hinge = max(0, 1 - margin)
                total_loss += hinge
                
                if margin >= 1:
                    # Точка правильно классифицирована и вне margin
                    # Градиент только от регуляризации
                    self.w -= self.lr * lam * self.w
                else:
                    # Точка нарушает margin
                    self.w -= self.lr * (lam * self.w - y_[i] * X[i])
                    self.b -= self.lr * (-y_[i])
            
            # Полный loss: регуляризация + средний hinge
            reg_loss = 0.5 * lam * np.dot(self.w, self.w)
            avg_hinge = total_loss / n_samples
            self.losses.append(reg_loss + avg_hinge)
        
        return self
    
    def decision_function(self, X):
        return np.dot(X, self.w) + self.b
    
    def predict(self, X):
        return np.sign(self.decision_function(X))


# --- Генерация линейно разделимых данных ---
X_lin, y_lin = make_blobs(n_samples=200, centers=2, random_state=42, cluster_std=1.5)
y_lin = np.where(y_lin == 0, -1, 1)  # [-1, +1]

# Нормализация
scaler = StandardScaler()
X_lin = scaler.fit_transform(X_lin)

# Обучение ручной реализации
svm_manual = LinearSVM(C=1.0, lr=0.01, n_iters=500)
svm_manual.fit(X_lin, y_lin)

preds = svm_manual.predict(X_lin)
acc = np.mean(preds == y_lin)
print(f"Точность ручной SVM: {acc:.3f}")
print(f"Веса w: {svm_manual.w}")
print(f"Bias b: {svm_manual.b:.4f}")

In [ ]:
# Визуализация сходимости и границы решения
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- График 1: Кривая потерь ---
axes[0].plot(svm_manual.losses, color='steelblue', linewidth=2)
axes[0].set_xlabel('Эпоха', fontsize=12)
axes[0].set_ylabel('Loss (Hinge + Reg)', fontsize=12)
axes[0].set_title('Кривая обучения — Ручной SVM', fontsize=13)
axes[0].grid(True, alpha=0.3)

# --- График 2: Decision Boundary ---
ax = axes[1]
x_min, x_max = X_lin[:, 0].min() - 0.5, X_lin[:, 0].max() + 0.5
y_min, y_max = X_lin[:, 1].min() - 0.5, X_lin[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))

Z = svm_manual.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Цветовой фон
ax.contourf(xx, yy, Z, levels=[-100, 0, 100], alpha=0.3, colors=['#FFAAAA', '#AAAAFF'])

# Граница решений (w·x + b = 0)
ax.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2.5, linestyles='-')

# Margin lines (w·x + b = ±1)
ax.contour(xx, yy, Z, levels=[-1, 1], colors=['red', 'blue'],
           linewidths=1.5, linestyles='--')

# Точки данных
colors = ['#FF4444' if label == -1 else '#4444FF' for label in y_lin]
ax.scatter(X_lin[:, 0], X_lin[:, 1], c=colors, edgecolors='k', s=50, alpha=0.8)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.set_title('Ручной Linear SVM — Decision Boundary\n(пунктир = margin ±1)', fontsize=13)

# Легенда
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='k', lw=2.5, label='Decision Boundary'),
    Line2D([0], [0], color='red', lw=1.5, linestyle='--', label='Margin -1'),
    Line2D([0], [0], color='blue', lw=1.5, linestyle='--', label='Margin +1'),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_manual_boundary.png', dpi=100, bbox_inches='tight')
plt.show()

## Sklearn SVM с линейным ядром

In [ ]:
def plot_svm_boundary(model, X, y, ax, title):
    """Универсальная функция для визуализации границы решения SVM."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min_, y_max_ = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min_, y_max_, 300),
    )

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")

    try:
        df = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contour(xx, yy, df, levels=[0], colors="k", linewidths=2.5)
        if hasattr(model, "kernel") and model.kernel == "linear":
            ax.contour(
                xx,
                yy,
                df,
                levels=[-1, 1],
                colors=["red", "blue"],
                linewidths=1.5,
                linestyles="--",
            )
    except Exception:
        pass

    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors="k", s=50, alpha=0.8, zorder=3)

    if hasattr(model, "support_vectors_"):
        sv = model.support_vectors_
        ax.scatter(
            sv[:, 0],
            sv[:, 1],
            s=200,
            linewidths=2.5,
            facecolors="none",
            edgecolors="gold",
            zorder=4,
            label=f"Опорные векторы ({len(sv)})",
        )
        ax.legend(fontsize=9, loc="upper right")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min_, y_max_)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Признак 1")
    ax.set_ylabel("Признак 2")
    ax.grid(True, alpha=0.3)


def fit_svm_with_split(X, y, *, kernel, test_size=0.3, random_state=42, **params):
    """Обучение SVM с фиксированным train/test split без утечки из test в scaler."""
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    X_all_sc = scaler.transform(X)

    model = SVC(kernel=kernel, **params)
    model.fit(X_train_sc, y_train)

    train_pred = model.predict(X_train_sc)
    test_pred = model.predict(X_test_sc)

    return {
        "model": model,
        "X_all_scaled": X_all_sc,
        "X_train_scaled": X_train_sc,
        "X_test_scaled": X_test_sc,
        "y_train": y_train,
        "y_test": y_test,
        "train_acc": accuracy_score(y_train, train_pred),
        "test_acc": accuracy_score(y_test, test_pred),
    }


# --- Данные: линейно разделимые (blobs) ---
X_blobs, y_blobs = make_blobs(n_samples=200, centers=2, random_state=42, cluster_std=1.5)
X_blobs = StandardScaler().fit_transform(X_blobs)

# Обучение sklearn SVM
svm_linear = SVC(kernel="linear", C=1.0)
svm_linear.fit(X_blobs, y_blobs)

print(f"Количество опорных векторов: {len(svm_linear.support_vectors_)}")
print(f"Опорные векторы по классам: {svm_linear.n_support_}")
print(f"Точность: {svm_linear.score(X_blobs, y_blobs):.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
plot_svm_boundary(
    svm_linear,
    X_blobs,
    y_blobs,
    ax,
    "Sklearn Linear SVM\nЗолотые окружности = опорные векторы",
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_linear_sklearn.png', dpi=100, bbox_inches="tight")
plt.show()

## Нелинейные данные: Make Moons

In [ ]:
# Генерация данных Moons
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

moons_linear = fit_svm_with_split(X_moons, y_moons, kernel="linear", C=1.0, random_state=42)
moons_rbf = fit_svm_with_split(X_moons, y_moons, kernel="rbf", C=1.0, gamma="scale", random_state=42)

print("=== Make Moons: отдельные обучающая и тестовая части ===")
print(
    f"Линейное ядро | train acc: {moons_linear['train_acc']:.3f} | "
    f"test acc: {moons_linear['test_acc']:.3f} | "
    f"SVs: {len(moons_linear['model'].support_vectors_)}"
)
print(
    f"RBF-ядро      | train acc: {moons_rbf['train_acc']:.3f} | "
    f"test acc: {moons_rbf['test_acc']:.3f} | "
    f"SVs: {len(moons_rbf['model'].support_vectors_)}"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_svm_boundary(
    moons_linear["model"],
    moons_linear["X_all_scaled"],
    y_moons,
    axes[0],
    (
        "Линейное ядро на Moons\n"
        f"Train/Test acc: {moons_linear['train_acc']:.3f}/{moons_linear['test_acc']:.3f}"
    ),
)
plot_svm_boundary(
    moons_rbf["model"],
    moons_rbf["X_all_scaled"],
    y_moons,
    axes[1],
    (
        "RBF-ядро на Moons\n"
        f"Train/Test acc: {moons_rbf['train_acc']:.3f}/{moons_rbf['test_acc']:.3f}"
    ),
)

plt.suptitle("Сравнение ядер на Make Moons без утечки из тестовой части", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_moons_comparison.png', dpi=100, bbox_inches="tight")
plt.show()

## Нелинейные данные: Make Circles

In [ ]:
# Генерация данных Circles
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

circles_linear = fit_svm_with_split(X_circles, y_circles, kernel="linear", C=1.0, random_state=42)
circles_rbf = fit_svm_with_split(X_circles, y_circles, kernel="rbf", C=1.0, gamma="scale", random_state=42)

print("=== Make Circles: отдельные обучающая и тестовая части ===")
print(
    f"Линейное ядро | train acc: {circles_linear['train_acc']:.3f} | "
    f"test acc: {circles_linear['test_acc']:.3f} | "
    f"SVs: {len(circles_linear['model'].support_vectors_)}"
)
print(
    f"RBF-ядро      | train acc: {circles_rbf['train_acc']:.3f} | "
    f"test acc: {circles_rbf['test_acc']:.3f} | "
    f"SVs: {len(circles_rbf['model'].support_vectors_)}"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_svm_boundary(
    circles_linear["model"],
    circles_linear["X_all_scaled"],
    y_circles,
    axes[0],
    (
        "Линейное ядро на Circles\n"
        f"Train/Test acc: {circles_linear['train_acc']:.3f}/{circles_linear['test_acc']:.3f}"
    ),
)
plot_svm_boundary(
    circles_rbf["model"],
    circles_rbf["X_all_scaled"],
    y_circles,
    axes[1],
    (
        "RBF-ядро на Circles\n"
        f"Train/Test acc: {circles_rbf['train_acc']:.3f}/{circles_rbf['test_acc']:.3f}"
    ),
)

plt.suptitle("Сравнение ядер на Make Circles без утечки из тестовой части", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_circles_comparison.png', dpi=100, bbox_inches="tight")
plt.show()

## Что меняет параметр `C`

In [ ]:
# Данные с некоторым перекрытием
X_soft, y_soft = make_blobs(n_samples=200, centers=2, random_state=42, cluster_std=2.5)
X_train_soft, X_test_soft, y_train_soft, y_test_soft = train_test_split(
    X_soft,
    y_soft,
    test_size=0.3,
    random_state=42,
    stratify=y_soft,
)

scaler_soft = StandardScaler()
X_train_soft_sc = scaler_soft.fit_transform(X_train_soft)
X_test_soft_sc = scaler_soft.transform(X_test_soft)
X_soft_sc = scaler_soft.transform(X_soft)

C_values = [0.01, 0.1, 1.0, 100.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

c_rows = []

for i, C in enumerate(C_values):
    model = SVC(kernel="linear", C=C)
    model.fit(X_train_soft_sc, y_train_soft)

    train_acc = accuracy_score(y_train_soft, model.predict(X_train_soft_sc))
    test_acc = accuracy_score(y_test_soft, model.predict(X_test_soft_sc))
    n_sv = len(model.support_vectors_)

    w_norm = np.linalg.norm(model.coef_[0])
    margin_width = 2.0 / w_norm if w_norm > 0 else float("inf")

    c_rows.append((C, train_acc, test_acc, n_sv, margin_width))

    title = (
        f"C = {C}\n"
        f"Train/Test acc: {train_acc:.3f}/{test_acc:.3f} | опорных: {n_sv}\n"
        f"Margin width ≈ {margin_width:.3f}"
    )
    plot_svm_boundary(model, X_soft_sc, y_soft, axes[i], title)

plt.suptitle(
    "Влияние параметра C на ширину Margin\n"
    "(метрики считаем на отдельной тестовой части, пунктиром показаны границы margin ±1)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_C_comparison.png', dpi=100, bbox_inches="tight")
plt.show()

print("\nСводная таблица:")
print(f"{'C':>8} | {'Train Acc':>10} | {'Test Acc':>9} | {'# SV':>8} | {'Ширина margin':>14}")
print("-" * 66)
for C, train_acc, test_acc, n_sv, margin_width in c_rows:
    print(f"{C:>8.2f} | {train_acc:>10.3f} | {test_acc:>9.3f} | {n_sv:>8d} | {margin_width:>14.4f}")

## Что меняет `gamma` в RBF-ядре

In [ ]:
X_gamma, y_gamma = make_moons(n_samples=300, noise=0.2, random_state=42)
X_train_gamma, X_test_gamma, y_train_gamma, y_test_gamma = train_test_split(
    X_gamma,
    y_gamma,
    test_size=0.3,
    random_state=42,
    stratify=y_gamma,
)

scaler_gamma = StandardScaler()
X_train_gamma_sc = scaler_gamma.fit_transform(X_train_gamma)
X_test_gamma_sc = scaler_gamma.transform(X_test_gamma)
X_gamma_sc = scaler_gamma.transform(X_gamma)

gamma_values = [0.1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for i, gamma in enumerate(gamma_values):
    model = SVC(kernel="rbf", C=1.0, gamma=gamma)
    model.fit(X_train_gamma_sc, y_train_gamma)

    train_acc = accuracy_score(y_train_gamma, model.predict(X_train_gamma_sc))
    test_acc = accuracy_score(y_test_gamma, model.predict(X_test_gamma_sc))
    n_sv = len(model.support_vectors_)

    title = (
        f"RBF gamma={gamma}\n"
        f"Train/Test acc: {train_acc:.3f}/{test_acc:.3f} | опорных: {n_sv}"
    )
    plot_svm_boundary(model, X_gamma_sc, y_gamma, axes[i], title)

plt.suptitle(
    "Влияние параметра gamma в RBF-ядре\n"
    "(малый gamma даёт широкое влияние, большой gamma делает ядро локальным; метрики считаем на тестовой части)",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_gamma_comparison.png', dpi=100, bbox_inches="tight")
plt.show()

## Почему именно эти точки становятся support vectors

In [ ]:
# Проверяем на практике: опорные векторы лежат на margin или внутри него
X_demo, y_demo = make_blobs(n_samples=140, centers=2, random_state=7, cluster_std=1.8)
X_demo = StandardScaler().fit_transform(X_demo)

svm_full = SVC(kernel="linear", C=1.0)
svm_full.fit(X_demo, y_demo)

sv_idx = svm_full.support_
support_mask = np.zeros(len(X_demo), dtype=bool)
support_mask[sv_idx] = True

y_signed = np.where(y_demo == 0, -1, 1)
decision_values = svm_full.decision_function(X_demo)
functional_margins = y_signed * decision_values

sv_margins = functional_margins[support_mask]
non_sv_margins = functional_margins[~support_mask]

margin_sv_count = np.sum(np.isclose(sv_margins, 1.0, atol=0.05))
violating_sv_count = np.sum(sv_margins < 1.0 - 0.05)
confidently_outside_count = np.sum(non_sv_margins > 1.0 + 0.05)

X_sv_only = X_demo[sv_idx]
y_sv_only = y_demo[sv_idx]

svm_sv_only = SVC(kernel="linear", C=1.0)
svm_sv_only.fit(X_sv_only, y_sv_only)

w_full = svm_full.coef_[0]
b_full = svm_full.intercept_[0]
w_sv = svm_sv_only.coef_[0]
b_sv = svm_sv_only.intercept_[0]
cos_sim = np.dot(w_full, w_sv) / (np.linalg.norm(w_full) * np.linalg.norm(w_sv))
agreement = np.mean(svm_full.predict(X_demo) == svm_sv_only.predict(X_demo))

print("=== Анализ выбранных опорных векторов ===")
print(f"Всего объектов: {len(X_demo)}")
print(f"Опорных векторов: {len(sv_idx)}")
print(f"SV на границе margin (y*f(x) ≈ 1): {margin_sv_count}")
print(f"SV-нарушители (y*f(x) < 1): {violating_sv_count}")
print(f"Не-SV далеко от границы (y*f(x) > 1): {confidently_outside_count}")
print(f"Минимальный margin среди SV: {sv_margins.min():.3f}")
print(f"Минимальный margin среди non-SV: {non_sv_margins.min():.3f}")

print("\n=== Что дают только опорные векторы ===")
print(f"Полная модель: w = {w_full}, b = {b_full:.4f}")
print(f"Только SVs:    w = {w_sv}, b = {b_sv:.4f}")
print(f"Косинусное сходство весов: {cos_sim:.6f}")
print(f"Совпадение предсказаний на всех данных: {agreement:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_svm_boundary(
    svm_full,
    X_demo,
    y_demo,
    axes[0],
    (
        "Полная модель\n"
        f"SV: {len(svm_full.support_vectors_)} | "
        f"margin SV: {margin_sv_count}, violators: {violating_sv_count}"
    ),
)

plot_svm_boundary(
    svm_sv_only,
    X_demo,
    y_demo,
    axes[1],
    (
        "Модель обучена только на SV\n"
        f"Prediction agreement: {agreement:.3f} | cos sim: {cos_sim:.4f}"
    ),
)

plt.suptitle(
    "Почему именно эти точки становятся support vectors:\n"
    "у них y*f(x) ≈ 1 или y*f(x) < 1, а у остальных точек запас по margin заметно больше",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_sv_proof.png', dpi=100, bbox_inches="tight")
plt.show()

## Сравнение на нескольких наборах данных

In [ ]:
datasets = [
    (*make_blobs(n_samples=200, centers=2, random_state=42, cluster_std=1.5), "Blobs (linear)"),
    (*make_moons(n_samples=300, noise=0.2, random_state=42), "Moons (nonlinear)"),
    (*make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42), "Circles (nonlinear)"),
]

kernels = [("linear", {"C": 1.0}), ("rbf", {"C": 1.0, "gamma": "scale"})]

fig, axes = plt.subplots(3, 2, figsize=(14, 18))

summary_rows = []

for row, (X_d, y_d, dname) in enumerate(datasets):
    for col, (kernel, params) in enumerate(kernels):
        result = fit_svm_with_split(
            X_d,
            y_d,
            kernel=kernel,
            random_state=42,
            **params,
        )

        model = result["model"]
        train_acc = result["train_acc"]
        test_acc = result["test_acc"]
        n_sv = len(model.support_vectors_)

        summary_rows.append((dname, kernel, train_acc, test_acc, n_sv))

        title = (
            f"{dname}\n"
            f"{kernel.upper()} | Train/Test acc: {train_acc:.3f}/{test_acc:.3f} | SVs: {n_sv}"
        )
        plot_svm_boundary(model, result["X_all_scaled"], y_d, axes[row][col], title)

plt.suptitle(
    "Итоговое сравнение: линейное ядро и RBF на разных данных\n"
    "(обе модели оцениваются на отдельном test split)",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'svm_final_comparison.png', dpi=100, bbox_inches="tight")
plt.show()

print("\n=== Сводка по качеству на тестовой части ===")
for dname, kernel, train_acc, test_acc, n_sv in summary_rows:
    print(
        f"{dname:20s} | {kernel.upper():6s} | "
        f"train={train_acc:.3f} | test={test_acc:.3f} | SVs={n_sv}"
    )

print("\n=== Короткие выводы ===")
print("1. Линейное ядро хорошо работает там, где классы и правда можно разделить прямой.")
print("2. RBF выигрывает на нелинейных наборах, потому что строит гибкую границу.")
print("3. Малый C обычно расширяет margin и чаще увеличивает число опорных векторов.")
print("4. Слишком большой gamma даёт слишком локальную границу и быстрее ведёт к переобучению.")
print("5. Опорные векторы лежат на margin или внутри него; именно они фиксируют положение границы.")

## Заключение

Главное по результатам:

1. Линейный SVM легко реализовать вручную через `Hinge Loss`, и это хороший способ почувствовать геометрию задачи.
2. На линейно разделимых данных линейное ядро обычно хватает с запасом.
3. На `moons` и `circles` уже видно, зачем нужен `RBF`: он строит нелинейную границу там, где прямая бессильна.
4. `C` регулирует жёсткость модели, а `gamma` задаёт локальность влияния точек в `RBF`.
5. Support vectors можно увидеть и глазами, и через величину `y * f(x)`: именно эти объекты и определяют итоговую границу.